In [135]:
!pip install pyspark


In [136]:
import os
from pyspark.sql import SparkSession

!rm -rf ./data_stream
!rm -rf ./checkpoint_dir

os.makedirs('data_stream', exist_ok=True)

spark = SparkSession.builder \
    .appName("HospitalMonitorColab") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark initialized and data_stream folder created.")


Spark initialized and data_stream folder created.


In [137]:
from pyspark.sql.functions import col, window, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import os

if os.path.exists("alerts.log"):
    os.remove("alerts.log")

schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("heart_rate", IntegerType(), True)
])

streaming_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(schema) \
    .load("./data_stream")


windowed_df = streaming_df \
    .withWatermark("timestamp", "1 minute") \
    .groupBy(
        col("patient_id"),
        window(col("timestamp"), "2 minutes")
    ) \
    .agg(avg("heart_rate").alias("avg_hr"))


high_hr_df = windowed_df.filter(col("avg_hr") > 100)

patient_state = {}

def evaluate_consecutive_windows(batch_df, batch_id):
    rows = batch_df.collect()
    for row in rows:
        pid = row["patient_id"]
        w_start = row["window"]["start"]
        w_end = row["window"]["end"]

        if pid in patient_state:
            prev_end = patient_state[pid]

            if w_start == prev_end:
                alert_msg = f"🚨 CLINICAL ALERT: Patient {pid} sustained elevated HR > 100 across 2 consecutive windows ending at {w_end}\n"

                with open("alerts.log", "a") as log_file:
                    log_file.write(alert_msg)
                print(alert_msg)


        patient_state[pid] = w_end


query = high_hr_df.writeStream \
    .outputMode("append") \
    .option("checkpointLocation", "./checkpoint_dir") \
    .foreachBatch(evaluate_consecutive_windows) \
    .start()

print("Streaming job is now running in the background...")


Streaming job is now running in the background...


In [138]:
import time
import os

csv1 = """patient_id,timestamp,heart_rate
P001,2023-11-01 10:00:15,105
P001,2023-11-01 10:01:30,110
P002,2023-11-01 10:00:45,75
"""

csv2 = """patient_id,timestamp,heart_rate
P001,2023-11-01 10:02:15,108
P001,2023-11-01 10:03:45,115
P002,2023-11-01 10:02:20,80
"""

csv3 = """patient_id,timestamp,heart_rate
P999,2023-11-01 10:10:00,70
"""

print("Ingesting Window 1...")
with open("data_stream/file1.csv", "w") as f: f.write(csv1)
time.sleep(5)

print("Ingesting Window 2...")
with open("data_stream/file2.csv", "w") as f: f.write(csv2)
time.sleep(5)

print("Triggering watermark to finalize windows...")
with open("data_stream/file3.csv", "w") as f: f.write(csv3)
time.sleep(10)

print("\n================ PIPELINE ALERTS ================\n")
if os.path.exists("alerts.log"):
    with open("alerts.log", "r") as log_file:
        print(log_file.read())
else:
    print("No alerts triggered or stream still catching up.")
print("=================================================")


Ingesting Window 1...
Ingesting Window 2...
Triggering watermark to finalize windows...

================ PIPELINE ALERTS ================

No alerts triggered or stream still catching up.


In [139]:
query.stop()
print("Streaming job safely stopped.")


ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/content/spark-3.5.1-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_1161/560682030.py", line 36, in evaluate_consecutive_windows
    rows = batch_df.collect()
           ^^^^^^^^^^^^^^^^^^
  File "/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/dataframe.py", line 1261, in collect
    sock_info = self._jdf.collectToPython()
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/spark-3.5.1-bin-hado

Streaming job safely stopped.


In [140]:
import os
import shutil
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# 1. CLEANUP: Crucial step to wipe old checkpoints so Spark reads the data fresh
for path in ["./data_stream", "./checkpoint_dir"]:
    if os.path.exists(path):
        shutil.rmtree(path)
os.makedirs('./data_stream', exist_ok=True)
if os.path.exists("alerts.log"):
    os.remove("alerts.log")

# 2. INITIALIZE SPARK
spark = SparkSession.builder.appName("Assignment").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 3. DEFINE STREAMING DATAFRAME
schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("heart_rate", IntegerType(), True)
])

streaming_df = spark.readStream.format("csv") \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(schema) \
    .load("./data_stream")

# 4. APPLY TUMBLING WINDOW & CONDITIONS
windowed_df = streaming_df \
    .withWatermark("timestamp", "1 minute") \
    .groupBy(col("patient_id"), window(col("timestamp"), "2 minutes")) \
    .agg(avg("heart_rate").alias("avg_hr"))

high_hr_df = windowed_df.filter(col("avg_hr") > 100)
patient_state = {}

def evaluate_consecutive_windows(batch_df, batch_id):
    # Order by time so evaluation is reliable even if batches merge
    rows = batch_df.orderBy("window.start").collect()
    for row in rows:
        pid = row["patient_id"]
        w_start = row["window"]["start"]
        w_end = row["window"]["end"]

        if pid in patient_state:
            if w_start == patient_state[pid]:
                msg = f"🚨 CLINICAL ALERT: Patient {pid} sustained elevated HR > 100 across 2 consecutive windows ending at {w_end}\n"
                with open("alerts.log", "a") as f: f.write(msg)
        patient_state[pid] = w_end

# 5. START STREAM
query = high_hr_df.writeStream.outputMode("append") \
    .option("checkpointLocation", "./checkpoint_dir") \
    .foreachBatch(evaluate_consecutive_windows) \
    .start()

# 6. SIMULATE DATA
csv1 = "patient_id,timestamp,heart_rate\nP001,2023-11-01 10:00:15,105\nP001,2023-11-01 10:01:30,110\nP002,2023-11-01 10:00:45,75\n"
csv2 = "patient_id,timestamp,heart_rate\nP001,2023-11-01 10:02:15,108\nP001,2023-11-01 10:03:45,115\nP002,2023-11-01 10:02:20,80\n"
csv3 = "patient_id,timestamp,heart_rate\nP999,2023-11-01 10:10:00,70\n"

print("Writing simulated streaming data...")
with open("data_stream/file1.csv", "w") as f: f.write(csv1)
time.sleep(1)
with open("data_stream/file2.csv", "w") as f: f.write(csv2)
time.sleep(1)
with open("data_stream/file3.csv", "w") as f: f.write(csv3)

print("Waiting for streaming pipeline to finish processing all batches...")
query.processAllAvailable() # <--- This forces Colab to wait until Spark completes the workload

print("\n================ PIPELINE ALERTS ================\n")
if os.path.exists("alerts.log"):
    with open("alerts.log", "r") as log_file:
        print(log_file.read().strip())
else:
    print("No alerts triggered or stream still catching up.")
print("\n=================================================")

# Safely stop query
query.stop()


Writing simulated streaming data...
Waiting for streaming pipeline to finish processing all batches...

================ PIPELINE ALERTS ================

🚨 CLINICAL ALERT: Patient P001 sustained elevated HR > 100 across 2 consecutive windows ending at 2023-11-01 10:04:00

